In [ ]:
########################################
#getting system arguments
import sys
def GetArg_dataName(default="Variables"):
    """
    #non-perturbations
    #Run One: python Tracked_Profiles.py Variables
    #Run Two: python Tracked_Profiles.py Entrainment
    #Run Three: python Tracked_Profiles.py PROCESSED_Entrainment (also can add _DivideMassFlux)
    #Run Four: python Tracked_Profiles.py W_Budgets
    #Run Five: python Tracked_Profiles.py QV_Budgets
    #Run Six: python Tracked_Profiles.py TH_Budgets

    #perturbations
    #Run One: python Tracked_Profiles.py Variables_Perturbations
    """
    # If run inside Jupyter, sys.argv will include ipykernel arguments
    if any("ipykernel_launcher" in arg for arg in sys.argv):
        print(f"Using default dataName: {default}")
        return default

    # If a user-specified argument exists, use it
    if len(sys.argv) > 1:
        out=sys.argv[1]
        print(f"Using argument dataName: {out}")
        return out

    return default

dataName = GetArg_dataName()

In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm
import copy

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import *

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=1)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData.res, ModelData.t_res, ModelData.Nz_str,
                                ModelData.Np_str, dataType="Tracking_Algorithms", dataName="Lagrangian_UpdraftTracking",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#data manager class (for saving data)
DataManager_TrackedProfiles = DataManager_Class(mainDirectory, scratchDirectory, ModelData.res, ModelData.t_res, ModelData.Nz_str,
                                ModelData.Np_str, dataType="Tracked_Profiles", dataName="Tracked_Ascent_Trajectories",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","2_Tracking_Algorithms"))
from CLASSES_TrackingAlgorithms import TrackingAlgorithms_DataLoading_Class, Results_InputOutput_Class, TrackedParcel_Loading_Class

In [ ]:
# IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","3_Tracked_Profiles"))
from CLASSES_TrackedProfiles import TrackedProfiles_DataLoading_CLASS

In [ ]:
#IMPORT FUNCTIONS

import sys
path=os.path.join(mainCodeDirectory,'Functions/')
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

In [ ]:
##############################################
#JOB ARRAY

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs(res,t_res):
    if res=='1km':
        if t_res=='5min':
            num_jobs=20
        elif t_res=='1min':
            num_jobs=100
    elif res=='250m': 
        if t_res=='1min':
            num_jobs=500
    return num_jobs
num_jobs = GetNumJobs(ModelData.res,ModelData.t_res)
SlurmJobArray = SlurmJobArray_Class(total_elements=ModelData.Np, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = SlurmJobArray.start_job; end_job = SlurmJobArray.end_job

def GetNumElements():
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job]
    return loop_elements
loop_elements = GetNumElements()

In [ ]:
##############################################
#DATA LOADING FUNCTIONS

In [ ]:
def Get_Lagrangian_Binary_Array_Data(ModelData, pIndices, varNames=None): 
    """
    Version for slicing a range of specific parcels
    """
    directory_Lagrangian_Binary_Array = f"/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/Code/OUTPUT/Variable_Calculation/LagrangianArrays/{ModelData.res}_{ModelData.t_res}_{ModelData.Nz_str}nz/Lagrangian_Binary_Array/"
    def limitParcels(ds): 
        ds = ds.isel(phony_dim_0=pIndices)
        if varNames is not None:
                ds = ds[varNames]
        return ds
    Lagrangian_Binary_Array_Data = OpenMultipleSingleTimes_LagrangianArray_JobArray(directory=directory_Lagrangian_Binary_Array,
                                                                                    ModelData=ModelData, start_job=None,end_job=None, 
                                                                                    limitParcels=limitParcels)
    return Lagrangian_Binary_Array_Data

# def Get_Lagrangian_Binary_Array_Data(ModelData, start_job,end_job):
#     """
#     Version for slicing a range of jobs
#     """
#     directory_Lagrangian_Binary_Array = f"/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/Code/OUTPUT/Variable_Calculation/LagrangianArrays/{ModelData.res}_{ModelData.t_res}_{ModelData.Nz_str}nz/Lagrangian_Binary_Array/"
#     Lagrangian_Binary_Array_Data,files = OpenMultipleSingleTimes_LagrangianArray_JobArray(directory_Lagrangian_Binary_Array, ModelData, start_job,end_job)
#     return Lagrangian_Binary_Array_Data

def GetSpatialData(Lagrangian_Binary_Array_Data):
    ds = Lagrangian_Binary_Array_Data.compute()

    # Now extract NumPy arrays
    W = ds['W'].data
    QCQI = ds['QCQI'].data
    parcel_z = ds['z'].data

    return W, QCQI, parcel_z

# def MakeTestArray():
#     trackedArray = np.zeros((5,3),dtype='int')
#     trackedArray[:,0]=[1,3,5,7,9] #p
#     trackedArray[:,1]=[80,90,100,110,120] #t1
#     trackedArray[:,2]=[85,96,107,118,129] #t2
#     return trackedArray

In [ ]:
##############################################
#COMPUTING FUNCTIONS

In [ ]:
def GetData(trackedArray):
    pIndices=trackedArray[:,0] #for example all pIndices for CL parcels
    varNames = ['W','QCQI','z']
    Lagrangian_Binary_Array_Data,_ = Get_Lagrangian_Binary_Array_Data(ModelData, pIndices, varNames=varNames)
    [W, QCQI, parcel_z] = GetSpatialData(Lagrangian_Binary_Array_Data)
    #varNames = add QV, THETA_v,THETA_e

    dataDictionary = {"W": W,
                      "QCQI": QCQI,
                      "parcel_z": parcel_z}
    return dataDictionary

def ComputeTrajectories(variableData,t1,t2,after):
    #making output array
    trajectoriesArray = np.full_like(variableData,fill_value=np.nan,dtype=float)

    #masking the final output array
    time_idx = np.arange(data.shape[0])[:, np.newaxis]
    mask = (time_idx >= t1) & (time_idx <= t2+after)
    
    #applying mask to output array
    trajectoriesArray[mask] = variableData[mask]

    return trajectoriesArray

In [ ]:
def RunCode(trackedArrays):

    parcelTypes=['CL','nonCL']
    parcelDepths=['SHALLOW','DEEP']
    
    #subset out parcelType and parcelDepth
    trajectoriesArrayDictionary = {pt: {pd: {} for pd in parcelDepths} for pt in parcelTypes}
    for parcelType in tqdm(parcelTypes):
        for parcelDepth in tqdm(parcelDepths, desc=f"Type: {parcelType}", leave=False):

            #subsetting data
            trackedArray = trackedArrays[parcelType][parcelDepth]
            t1 = trackedArray[:, 1].astype(int);t2 = trackedArray[:, 2].astype(int)
            after = trackedArray[:, 3].astype(int)
        
            # Data Calculations
            # loading variable data dictionary
            tqdm.write(f"Getting Data")
            dataDictionary = GetData(trackedArray)
            
            #getting data
            varNames = ['parcel_z','W','QCQI'] #...
            for varName in tqdm(varNames, desc="Subsetting Variable", leave=False):
                variableData = dataDictionary[varName]
                
                #computing trajectories for each variable
                trajectoriesArray = ComputeTrajectories(variableData,t1,t2,after)
                trajectoriesArrayDictionary[parcelType][parcelDepth][varName] = trajectoriesArray

    return trajectoriesArrayDictionary

In [ ]:
##############################################
#COMPUTING
running=True #KEEP TRUE WHEN JOBARRAY IS RUNNING
# running=False 

In [ ]:
if running:
    #Loading in Tracked Parcels Info
    # trackedArray = MakeTestArray() #*testing
    trackedArrays,_ = TrackedParcel_Loading_Class.LoadingSubsetParcelData(ModelData,DataManager,Results_InputOutput_Class)

    trajectoriesArrayDictionary = RunCode(trackedArrays)
    TrackedProfiles_DataLoading_CLASS.SaveProfile(ModelData,DataManager_TrackedProfiles, trajectoriesArrayDictionary, dataName, t='all_times')

In [ ]:
##############################################
#PLOTTING FUNCTIONS

In [ ]:
#Plotting All Trajectories

# def PlotTrajectories(trajectoriesArray,zArray): #simple version, not colored by z
#     time=np.arange(ModelData.Ntime)*ModelData.dt/3600+6
#     plt.plot(time, trajectoriesArray)
#     plt.xlim(time[0],time[-1])
#     plt.ylim(bottom=0) #for z
# PlotTrajectories(trajectoriesArray)

from matplotlib.collections import LineCollection
def PlotTrajectories(trajectoriesArray, zArray, cmap='turbo', yLabel=None,
                     vmaxMethod='nanmax',roundingNumber=1000,
                     fig=None,axis=None):
    time = np.arange(ModelData.Ntime) * ModelData.dt / 3600 + 6
    if fig is None:
        fig, axis = plt.subplots() # Better to create fig/ax explicitly

    # 1. Setup color scaling
    if vmaxMethod == 'nanmax':
        vmaxValue = np.nanmax(zArray)
    elif vmaxMethod == 'percentile99':
        vmaxValue = np.nanpercentile(zArray,99)
    if roundingNumber is not None:
        vmaxValue = np.ceil(vmaxValue/roundingNumber)*roundingNumber

    norm = plt.Normalize(vmin=0, vmax=vmaxValue)

    for i in range(trajectoriesArray.shape[1]):
        y = trajectoriesArray[:, i]
        z = zArray[:, i]
        
        mask = ~np.isnan(y)
        if not np.any(mask): continue
        
        t_m, y_m, z_m = time[mask], y[mask], z[mask]

        # 2. Reshape into segments
        points = np.array([t_m, y_m]).T.reshape(-1, 1, 2)
        segments = np.concatenate([points[:-1], points[1:]], axis=1)

        # 3. Create and add collection
        lc = LineCollection(segments, cmap=cmap, norm=norm, alpha=0.6)
        lc.set_array(z_m) 
        axis.add_collection(lc)

    # 4. Final touches
    axis.set_xlim(time[0], time[-1])
    axis.set_xlabel('time (hrs)')
    if yLabel is not None:
        axis.set_ylabel(yLabel)
    
    # Use axis.autoscale_view() or manual limits
    if not np.all(np.isnan(trajectoriesArray)):
        axis.set_ylim(0, np.nanmax(trajectoriesArray) * 1.1)
    
    # FIXED: Tell the colorbar which axes to use (ax=axis)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    fig.colorbar(sm, ax=axis, label='z (m)', extend='max')

In [ ]:
#Plotting All Trajectories

# from matplotlib.colors import LogNorm, Normalize
# def PlotDensity(trajectoriesArray,zArray, nBins=40, yLabel=None,
#                 fig=None, axis=None, 
#                 logNorm=False, lowerThreshold=None):
#     """
#     #Plot Density By Count
#     """
#     if fig is None: fig, axis = plt.subplots()

#     # 1. Flatten data
#     time = np.arange(ModelData.Ntime) * ModelData.dt / 3600 + 6
#     t_mesh = np.broadcast_to(time[:, None], trajectoriesArray.shape)
    
#     mask = ~np.isnan(trajectoriesArray)
#     if lowerThreshold is not None:
#         mask = mask & (trajectoriesArray > lowerThreshold)
#     t_flat, z_flat = t_mesh[mask], trajectoriesArray[mask]

#     # 2. Fix the Norm
#     # We want to scale the COLOR based on the number of particles (counts)
#     if logNorm:
#         norm = LogNorm() # Automatically scales based on count density
#     else:
#         norm = Normalize()

#     # 3. Plot
#     counts, xedges, yedges, im = axis.hist2d(
#         t_flat, z_flat, bins=[ModelData.Ntime // 2, nBins], 
#         cmap='turbo', norm=norm, cmin=1
#     )
    
#     fig.colorbar(im, ax=axis, label='Particle Count')
    
#     axis.set_xlabel('Time (h)')
#     if yLabel is not None: axis.set_ylabel(yLabel)
#     if not np.all(np.isnan(trajectoriesArray)):
#         axis.set_ylim(0, np.nanmax(trajectoriesArray) * 1.1)
    
#     # Only set bottom=0 for variables like height or QCQI
#     if np.nanmin(z_flat) >= 0:
#         axis.set_ylim(bottom=0)

# def PlotDensity(trajectoriesArray, zArray, nBins=40, yLabel=None,
#                 fig=None, axis=None, logNorm=False, lowerThreshold=None,
#                 vmaxMethod='nanmax',roundingNumber=1000):
#     """
#     #Plot Density By Height
#     #2D Histogram Version
#     """
#     if fig is None: fig, axis = plt.subplots()

#     if vmaxMethod == 'nanmax':
#         vmaxValue = np.nanmax(zArray)
#     elif vmaxMethod == 'percentile99':
#         vmaxValue = np.nanpercentile(zArray,99)
#     if roundingNumber is not None:
#         vmaxValue = np.ceil(vmaxValue/roundingNumber)*roundingNumber

#     # 1. Flatten data
#     time = np.arange(ModelData.Ntime) * ModelData.dt / 3600 + 6
#     t_mesh = np.broadcast_to(time[:, None], trajectoriesArray.shape)
    
#     mask = ~np.isnan(trajectoriesArray) & ~np.isnan(zArray)
#     if lowerThreshold is not None:
#         mask = mask & (trajectoriesArray > lowerThreshold)
        
#     t_flat = t_mesh[mask]
#     y_flat = trajectoriesArray[mask]
#     z_flat = zArray[mask]

#     # 2. Manually calculate the Binned Mean
#     # Define bins
#     x_bins = np.linspace(time[0], time[-1], ModelData.Ntime // 2)
#     y_bins = np.linspace(np.nanmin(y_flat), np.nanmax(y_flat), nBins)

#     # Sum of Z values in each bin
#     z_sum, x_edges, y_edges = np.histogram2d(t_flat, y_flat, bins=[x_bins, y_bins], weights=z_flat)
    
#     # Count of particles in each bin
#     counts, _, _ = np.histogram2d(t_flat, y_flat, bins=[x_bins, y_bins])

#     # Calculate Average (handle division by zero where counts=0)
#     with np.errstate(divide='ignore', invalid='ignore'):
#         z_mean = z_sum / counts

#     # 3. Plot using pcolormesh (which creates the 'hist2d' look)
#     # Note: We transpose z_mean because pcolormesh expects (Y, X) or similar alignment
#     im = axis.pcolormesh(x_edges, y_edges, z_mean.T, cmap='turbo', shading='auto',
#                         vmin=0, vmax=vmaxValue)
    
#     # 4. Final touches
#     fig.colorbar(im, ax=axis, label='z (m)')
    
#     axis.set_xlabel('Time (h)')
#     if yLabel is not None: axis.set_ylabel(yLabel)
#     axis.set_ylim(0, np.nanmax(trajectoriesArray) * 1.1)

def PlotDensity(trajectoriesArray, zArray, nBins=40, yLabel=None,
                fig=None, axis=None, logNorm=False, lowerThreshold=None,
                 vmaxMethod='nanmax',roundingNumber=1000):
    """
    #Plot Density By Height
    #HexBin Version
    """
    if fig is None: fig, axis = plt.subplots()

    if vmaxMethod == 'nanmax':
        vmaxValue = np.nanmax(zArray)
    elif vmaxMethod == 'percentile99':
        vmaxValue = np.nanpercentile(zArray,99)
    if roundingNumber is not None:
        vmaxValue = np.ceil(vmaxValue/roundingNumber)*roundingNumber

    # 1. Flatten data
    time = np.arange(ModelData.Ntime) * ModelData.dt / 3600 + 6
    t_mesh = np.broadcast_to(time[:, None], trajectoriesArray.shape)
    
    # Create mask for valid data and thresholds
    mask = ~np.isnan(trajectoriesArray) & ~np.isnan(zArray)
    if lowerThreshold is not None:
        mask = mask & (trajectoriesArray > lowerThreshold)
        
    t_flat = t_mesh[mask]
    y_flat = trajectoriesArray[mask]
    z_flat = zArray[mask] # The values we want to average for the color

    # 2. Use hexbin with 'C' parameter
    # C=z_flat tells hexbin to aggregate height values
    # reduce_C_function=np.nanmean calculates the average height in each bin
    hb = axis.hexbin(
        t_flat, y_flat, C=z_flat, 
        reduce_C_function=np.nanmean,
        gridsize=(ModelData.Ntime // 2, nBins),
        cmap='turbo', 
        mincnt=1, # Don't plot empty bins
        vmin=0,
        vmax=vmaxValue
    )
    
    # 3. Final touches
    fig.colorbar(hb, ax=axis, label='z (m)', extend='max')
    
    axis.set_xlabel('Time (h)')
    if yLabel is not None: axis.set_ylabel(yLabel)
    
    if not np.all(np.isnan(trajectoriesArray)):
        axis.set_ylim(0, np.nanmax(trajectoriesArray) * 1.1)
    
    if np.nanmin(y_flat) >= 0:
        axis.set_ylim(bottom=0)

def CombinedTrajectoryPlot(trajectoriesArrayDictionary,
                           parcelType,parcelDepth):
    fig, axes = plt.subplots(len(varNames), 2, figsize=(10, 4 * len(varNames)), gridspec_kw={"wspace":0.4})
    trajectoriesArray=trajectoriesArrayDictionary[parcelType][parcelDepth]
    zArray=trajectoriesArray['parcel_z']
    
    varName = 'parcel_z'
    axis=axes[0,0]
    # PlotTrajectories(trajectoriesArray[varName],zArray,yLabel=varName,
    #                  fig=fig,axis=axis)
    axis=axes[0,1]
    PlotDensity(trajectoriesArray[varName],zArray,yLabel=varName,
                fig=fig,axis=axis,
                logNorm=False)
    
    varName = 'W'
    axis=axes[1,0]
    # PlotTrajectories(trajectoriesArray[varName],zArray,yLabel=varName,
    #                  fig=fig,axis=axis)
    axis=axes[1,1]
    PlotDensity(trajectoriesArray[varName],zArray,yLabel=varName,
                fig=fig,axis=axis,
                logNorm=True,lowerThreshold=0.1)
    
    varName = 'QCQI'
    axis=axes[2,0]
    # PlotTrajectories(trajectoriesArray[varName],zArray,yLabel=varName,
    #                  fig=fig,axis=axis)
    axis=axes[2,1]
    PlotDensity(trajectoriesArray[varName],zArray,yLabel=varName,
                fig=fig,axis=axis,
                logNorm=True,lowerThreshold=1e-6)

    return fig

def CombinedTrajectoryPlot(trajectoriesArrayDictionary,
                           parcelType,parcelDepth):
    fig, axes = plt.subplots(len(varNames), 2, figsize=(10, 4 * len(varNames)), gridspec_kw={"wspace":0.4})
    trajectoriesArray=trajectoriesArrayDictionary[parcelType][parcelDepth]
    zArray=trajectoriesArray['parcel_z']
    
    varName = 'parcel_z'
    axis=axes[0,0]
    PlotTrajectories(trajectoriesArray[varName],zArray,yLabel=varName,
                     fig=fig,axis=axis)
    axis=axes[0,1]
    PlotDensity(trajectoriesArray[varName],zArray,yLabel=varName,
                fig=fig,axis=axis,
                logNorm=False)
    
    varName = 'W'
    axis=axes[1,0]
    PlotTrajectories(trajectoriesArray[varName],zArray,yLabel=varName,
                     fig=fig,axis=axis)
    axis=axes[1,1]
    PlotDensity(trajectoriesArray[varName],zArray,yLabel=varName,
                fig=fig,axis=axis,
                logNorm=True,lowerThreshold=0.1)
    
    varName = 'QCQI'
    axis=axes[2,0]
    PlotTrajectories(trajectoriesArray[varName],zArray,yLabel=varName,
                     fig=fig,axis=axis)
    axis=axes[2,1]
    PlotDensity(trajectoriesArray[varName],zArray,yLabel=varName,
                fig=fig,axis=axis,
                logNorm=True,lowerThreshold=1e-6)

    return fig

In [ ]:
#Plotting All Trajectories Mean

def PlotTrajectories_Mean(trajectoriesArray_mean,
                          yLabel=None,color='green',linestyle='-',
                          fig=None,axis=None): #mean version, may be useful later

    if fig is None:
        fig,axis=plt.subplots()
    time=np.arange(ModelData.Ntime)*ModelData.dt/3600+6
    axis.plot(time, trajectoriesArray_mean,color=color,linestyle=linestyle)
    axis.set_xlim(time[0],time[-1])
    axis.set_ylim(bottom=0) #for z

    axis.set_xlabel('Time (hrs)')
    if yLabel is not None: axis.set_ylabel(yLabel)

    if not np.all(np.isnan(trajectoriesArray_mean)):
        axis.set_ylim(0, np.nanmax(trajectoriesArray_mean) * 1.1)
import warnings
def CombinedTrajectoryMeanPlot(trajectoriesArrayDictionary,
                               parcelTypes,parcelDepth):

    fig, axes = plt.subplots(len(varNames), 1, figsize=(10, 4 * len(varNames)), gridspec_kw={"wspace":0.4})
    color='green' if parcelDepth == "SHALLOW" else 'blue'
    
    for count,parcelType in enumerate(parcelTypes):
        linestyle='-' if parcelType == "CL" else '--'

        trajectoriesArray=trajectoriesArrayDictionary[parcelType][parcelDepth]
        zArray=trajectoriesArray['parcel_z']

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            varName = 'parcel_z'
            axis=axes[0]
            PlotTrajectories_Mean(np.nanmean(trajectoriesArray[varName],axis=1),
                                  yLabel=varName, color=color,linestyle=linestyle,
                                  fig=fig,axis=axis)
    
            varName = 'W'
            axis=axes[1]
            PlotTrajectories_Mean(np.nanmean(trajectoriesArray[varName],axis=1),
                                  yLabel=varName, color=color,linestyle=linestyle,
                                  fig=fig,axis=axis)
            
            varName = 'QCQI'
            axis=axes[2]
            PlotTrajectories_Mean(np.nanmean(trajectoriesArray[varName],axis=1),
                                  yLabel=varName, color=color,linestyle=linestyle,
                                  fig=fig,axis=axis)

    return fig

In [ ]:
def CalculateAverageParcelLifeCyle(data):
    # data.shape is (133, 4230)
    n_times, n_parcels = data.shape
    
    # Create an empty container of the same shape filled with NaNs
    aligned_data = np.full((n_times, n_parcels), np.nan)
    
    for i in range(n_parcels):
        parcel_slice = data[:, i]
        
        # Find where the data is NOT nan
        mask = ~np.isnan(parcel_slice)
        
        if np.any(mask):
            # Extract only the "burst" of real data
            valid_data = parcel_slice[mask]
            
            # Place this burst at the beginning of our aligned array
            # This makes "Model Time X" become "Time Since Birth 0"
            aligned_data[:len(valid_data), i] = valid_data
            
    # Calculate the mean across parcels for each "relative" time step
    # As time increases, np.nanmean ignores the NaNs from shorter parcels
    mean_profile = np.nanmean(aligned_data, axis=1)
    # mean_profile = np.nanmax(aligned_data, axis=1)
    # mean_profile = np.nanpercentile(aligned_data,99, axis=1)
    
    # Calculate the count of active parcels at each relative time step
    counts = np.sum(~np.isnan(aligned_data), axis=1)

    return aligned_data,mean_profile, counts


def PlotAverageParcelLifeCyle_Mean(aligned,mean,counts,varName,
                                   label,color,linestyle,
                                   fig=None,axis=None, yLabel=None):
    
    # 2. Calculate Standard Error (Uncertainty of the mean)
    std = np.nanstd(aligned, axis=1); sem = std / np.sqrt(counts)
    mask = counts >= 1
    
    # 3. Plotting
    if fig is None:
        fig, axis = plt.subplots(figsize=(8, 5))
    
    t = np.arange(len(mean))*ModelData.dt/60    
    
    # Plot CL
    axis.plot(t[mask], mean[mask], color=color,linestyle=linestyle,label=label)
    axis.fill_between(t[mask], mean[mask] - sem[mask], 
                    mean[mask] + sem[mask], alpha=0.2)
   
    axis.set_xlabel("Minutes Since Birth"); 
    if yLabel is not None: axis.set_ylabel(yLabel)
    axis.legend()
    axis.set_title("Average Parcel Life Cycle")
    axis.set_xlim(left=0);#axis.set_ylim(bottom=0)
    
    if varName=="W":
        axhlineValue = 0.1
        axis.axhline(axhlineValue,color='gray',linestyle='--',zorder=-50)
    elif varName=="QCQI":
        axhlineValue = 1e-6
        axis.axhline(axhlineValue,color='gray',linestyle='--',zorder=-50)
        


def CombinedAverageParcelLifeCyleMeanPlot(trajectoriesArrayDictionary,
                               parcelTypes,parcelDepth):

    fig, axes = plt.subplots(len(varNames), 1, figsize=(10, 4 * len(varNames)), gridspec_kw={"hspace":0.4})
    color='green' if parcelDepth == "SHALLOW" else 'blue'
    
    for count,parcelType in enumerate(parcelTypes):
        linestyle='-' if parcelType == "CL" else '--'
        label=f'{parcelType} {parcelDepth}'


        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            varName = 'parcel_z'
            axis=axes[0]
            aligned, mean, counts = CalculateAverageParcelLifeCyle(\
                trajectoriesArrayDictionary[parcelType][parcelDepth][varName])
            PlotAverageParcelLifeCyle_Mean(aligned,mean,counts,varName,
                                           fig=fig,axis=axis, yLabel=varName,
                                           label=label,color=color,linestyle=linestyle)
    
            varName = 'W'
            axis=axes[1]
            aligned, mean, counts = CalculateAverageParcelLifeCyle(\
                trajectoriesArrayDictionary[parcelType][parcelDepth][varName])
            PlotAverageParcelLifeCyle_Mean(aligned,mean,counts,varName,
                                           fig=fig,axis=axis, yLabel=varName,
                                           label=label,color=color,linestyle=linestyle)
            
            varName = 'QCQI'
            axis=axes[2]
            aligned, mean, counts = CalculateAverageParcelLifeCyle(\
                trajectoriesArrayDictionary[parcelType][parcelDepth][varName])
            PlotAverageParcelLifeCyle_Mean(aligned,mean,counts,varName,
                                           fig=fig,axis=axis, yLabel=varName,
                                           label=label,color=color,linestyle=linestyle)

    return fig

In [ ]:
##############################
#PLOTTING

In [ ]:
trajectoriesArrayDictionary = TrackedProfiles_DataLoading_CLASS.LoadProfile(ModelData,DataManager_TrackedProfiles, dataName, t='all_times')

In [ ]:
fig=CombinedTrajectoryPlot(trajectoriesArrayDictionary,
                           parcelType="CL",parcelDepth="SHALLOW")
fig=CombinedTrajectoryPlot(trajectoriesArrayDictionary,
                           parcelType="nonCL",parcelDepth="SHALLOW")

In [ ]:
fig=CombinedTrajectoryPlot(trajectoriesArrayDictionary,
                           parcelType="CL",parcelDepth="DEEP")
fig=CombinedTrajectoryPlot(trajectoriesArrayDictionary,
                           parcelType="nonCL",parcelDepth="DEEP")

In [ ]:
fig=CombinedTrajectoryMeanPlot(trajectoriesArrayDictionary,
                               parcelTypes=["CL","nonCL"],parcelDepth="SHALLOW")

In [ ]:
fig=CombinedTrajectoryMeanPlot(trajectoriesArrayDictionary,
                               parcelTypes=["CL","nonCL"],parcelDepth="DEEP")

In [ ]:
fig = CombinedAverageParcelLifeCyleMeanPlot(trajectoriesArrayDictionary,
                                            parcelTypes=['CL','nonCL'],parcelDepth='SHALLOW')

In [ ]:
fig = CombinedAverageParcelLifeCyleMeanPlot(trajectoriesArrayDictionary,
                                            parcelTypes=['CL','nonCL'],parcelDepth='DEEP')

In [ ]:
############################
#future: save plots

In [ ]:
############################
#TESTING

In [ ]:
# #testing min parcel ascent initial W
# a=trajectoriesArrayDictionary['CL']['SHALLOW']['W']

# lst=[]
# for i in range(a.shape[1]):
#     b=a[:,i]
#     c=b[~np.isnan(b)][1]
#     lst.append(c)

# d=np.array(lst)
# d[d<0]